In [2]:
%pip install opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 217.1 kB/s eta 0:03:03
    --------------------------------------- 0.5/40.2 MB 217.1 kB/s eta 0:03:03
    --------------------------------------- 0.5/40.2 MB 217.1 kB/s eta 0:03:03
    --------------------------------------- 0.8/40.2 MB 268.6 k

# SG-1 DC Project — Task 3: Real-Time Inference (Sliding Window)



In [8]:
# ── Cell 1: Imports & Config ─────────────────────────────────────────────────

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from collections import deque
from torchvision import transforms
import time

CFG = {
    "cnn_weights" : "AI project/asl_cnn_weights.pth",
    "lstm_weights": "AI project/asl_lstm_weights.pth",
    "img_size"    : 128,       # must match training
    "seq_len"     : 16,        # sliding window size
    "feature_dim" : 512,
    "lstm_hidden" : 512,
    "lstm_layers" : 2,
    "cam_index"   : 0,         # change to 1,2... if default cam is wrong
    "conf_thresh" : 0.4,       # min softmax confidence to display prediction
    "display_w"   : 960,       # window width
    "display_h"   : 540,       # window height
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# Preprocessing — identical to Task 1 & 2
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

Device : cpu


In [4]:
# ── Cell 2: Model Definitions (identical to Task 1 & 2) ──────────────────────

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)), nn.ReLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, pool=False):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, stride=stride,
                              padding=kernel//2, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()
    def forward(self, x):
        return self.pool(F.relu(self.bn(self.conv(x)), inplace=True))


class ResBlock(nn.Module):
    def __init__(self, channels, se=True):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )
        self.se = SEBlock(channels) if se else nn.Identity()
    def forward(self, x):
        return F.relu(self.se(self.conv2(self.conv1(x))) + x, inplace=True)


class ASL_CNN(nn.Module):
    def __init__(self, num_classes=26, feature_dim=512, dropout=0.4):
        super().__init__()
        self.stage0 = ConvBlock(3, 32, kernel=5, pool=True)
        self.stage1 = nn.Sequential(ConvBlock(32, 64, pool=True),
                                    ResBlock(64), ResBlock(64), nn.Dropout2d(0.10))
        self.stage2 = nn.Sequential(ConvBlock(64, 128, pool=True),
                                    ResBlock(128), ResBlock(128),
                                    nn.Conv2d(128,128,1), nn.BatchNorm2d(128),
                                    nn.ReLU(inplace=True), nn.Dropout2d(0.15))
        self.stage3 = nn.Sequential(ConvBlock(128, 256, pool=True),
                                    ResBlock(256), nn.Dropout2d(0.20))
        self.stage4 = ConvBlock(256, feature_dim, pool=True)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout*0.5), nn.Linear(256, num_classes)
        )
        self.feature_dim = feature_dim

    def forward_features(self, x):
        x = self.stage0(x); x = self.stage1(x); x = self.stage2(x)
        x = self.stage3(x); x = self.stage4(x)
        return self.gap(x).flatten(1)

    def forward(self, x):
        return self.head(self.forward_features(x))


class ASL_LSTM(nn.Module):
    def __init__(self, cnn, feature_dim, hidden_size, num_layers,
                 num_classes, lstm_dropout=0.4, fc_dropout=0.4):
        super().__init__()
        self.cnn         = cnn
        self.feature_dim = feature_dim
        self.lstm = nn.LSTM(
            input_size=feature_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Dropout(fc_dropout), nn.Linear(hidden_size, 256),
            nn.ReLU(inplace=True), nn.Dropout(fc_dropout*0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        """x: (1, T, C, H, W) — batch size is always 1 at inference"""
        B, T, C, H, W = x.shape
        feats = self.cnn.forward_features(x.view(B*T, C, H, W))  # (T, 512)
        feats = feats.view(B, T, self.feature_dim)                # (1, T, 512)
        lstm_out, _ = self.lstm(feats)                            # (1, T, hidden)
        last = self.layer_norm(lstm_out[:, -1, :])                # (1, hidden)
        return self.head(last)                                    # (1, num_classes)


print("Model classes defined.")

Model classes defined.


In [10]:
# ── Cell 3: Load Weights ─────────────────────────────────────────────────────

# ── Load Task 2 checkpoint (has class2idx, num_classes, LSTM state) ──
lstm_ckpt   = torch.load(CFG["lstm_weights"], map_location=DEVICE)
class2idx   = lstm_ckpt["class2idx"]
idx2class   = lstm_ckpt["idx2class"]
NUM_CLASSES = lstm_ckpt["num_classes"]

# Make sure idx2class keys are ints (JSON serialization turns them to strings)
idx2class = {int(k): v for k, v in idx2class.items()}

print(f"Classes ({NUM_CLASSES}): {[idx2class[i] for i in range(NUM_CLASSES)]}")

# ── Build CNN backbone and load Task 1 weights ──
cnn_backbone = ASL_CNN(num_classes=26, feature_dim=CFG["feature_dim"]).to(DEVICE)
cnn_ckpt     = torch.load(CFG["cnn_weights"], map_location=DEVICE)
cnn_backbone.load_state_dict(cnn_ckpt["model_state"])
for p in cnn_backbone.parameters():
    p.requires_grad = False
cnn_backbone.eval()
print(f"CNN loaded (epoch {cnn_ckpt['epoch']}) and frozen.")

# ── Build LSTM model and load Task 2 weights ──
model = ASL_LSTM(
    cnn         = cnn_backbone,
    feature_dim = CFG["feature_dim"],
    hidden_size = CFG["lstm_hidden"],
    num_layers  = CFG["lstm_layers"],
    num_classes = NUM_CLASSES,
).to(DEVICE)

model.load_state_dict(lstm_ckpt["model_state"])
model.eval()
print(f"LSTM loaded (epoch {lstm_ckpt['epoch']}, val_acc={lstm_ckpt['val_acc']*100:.1f}%).")
print("\nModel ready for inference.")

Classes (31): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'HELLO', 'I', 'J', 'K', 'L', 'M', 'N', 'NO', 'O', 'P', 'Q', 'R', 'S', 'SORRY', 'T', 'THANKYOU', 'U', 'V', 'W', 'X', 'Y', 'YES', 'Z']
CNN loaded (epoch 4) and frozen.
LSTM loaded (epoch 60, val_acc=93.3%).

Model ready for inference.


In [11]:
# ── Cell 4: Inference Helpers ────────────────────────────────────────────────

def preprocess_frame(bgr_frame):
    """
    Convert a raw OpenCV BGR frame to a normalised tensor.
    Returns shape: (3, img_size, img_size)
    """
    rgb   = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    img   = Image.fromarray(rgb)
    return preprocess(img)   # (3, 128, 128)


def predict_from_window(frame_buffer):
    """
    Run one forward pass on the current 16-frame window.

    frame_buffer : deque of tensors, each (3, H, W)
    Returns      : (predicted_label: str, confidence: float, all_probs: tensor)
    """
    seq = torch.stack(list(frame_buffer))          # (16, 3, H, W)
    seq = seq.unsqueeze(0).to(DEVICE)              # (1, 16, 3, H, W)

    with torch.no_grad():
        logits = model(seq)                        # (1, num_classes)
        probs  = F.softmax(logits, dim=1)[0]       # (num_classes,)

    conf, idx = probs.max(0)
    label     = idx2class[idx.item()]
    return label, conf.item(), probs.cpu()


def draw_overlay(frame, label, confidence, buffer_size, fps,
                 top_preds, conf_thresh):
    """
    Draw prediction overlay on the frame in-place.
    Includes: main prediction, confidence bar, top-3 predictions, buffer fill, FPS.
    """
    h, w = frame.shape[:2]

    # ── Semi-transparent dark panel on the left ──
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (320, h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    # ── Main prediction ──
    active = confidence >= conf_thresh and buffer_size == CFG["seq_len"]
    color  = (0, 255, 120) if active else (100, 100, 100)

    cv2.putText(frame, "Prediction", (15, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(frame, label if active else "...",
                (15, 90), cv2.FONT_HERSHEY_DUPLEX, 2.2, color, 3, cv2.LINE_AA)

    # ── Confidence bar ──
    bar_x, bar_y, bar_w, bar_h = 15, 105, 280, 14
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x+bar_w, bar_y+bar_h), (60,60,60), -1)
    filled = int(bar_w * confidence)
    bar_color = (0, 220, 100) if confidence >= conf_thresh else (0, 140, 220)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x+filled, bar_y+bar_h), bar_color, -1)
    cv2.putText(frame, f"{confidence*100:.1f}%", (bar_x + bar_w + 8, bar_y + 11),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA)

    # ── Top-3 predictions ──
    cv2.putText(frame, "Top-3", (15, 145),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1, cv2.LINE_AA)
    for rank, (lbl, prob) in enumerate(top_preds[:3]):
        y_pos = 168 + rank * 28
        cv2.putText(frame, f"{rank+1}. {lbl:<10} {prob*100:5.1f}%",
                    (15, y_pos), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (220, 220, 220), 1, cv2.LINE_AA)

    # ── Buffer fill indicator ──
    buf_frac = buffer_size / CFG["seq_len"]
    buf_color = (0, 200, 255) if buffer_size < CFG["seq_len"] else (0, 255, 120)
    cv2.putText(frame, f"Buffer  {buffer_size:2d}/{CFG['seq_len']}",
                (15, h - 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150,150,150), 1, cv2.LINE_AA)
    cv2.rectangle(frame, (15, h-40), (295, h-26), (60,60,60), -1)
    cv2.rectangle(frame, (15, h-40), (15 + int(280*buf_frac), h-26), buf_color, -1)

    # ── FPS ──
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 110, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180,180,180), 1, cv2.LINE_AA)

    # ── Press Q hint ──
    cv2.putText(frame, "Press Q to quit", (w - 175, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120,120,120), 1, cv2.LINE_AA)

    return frame


print("Inference helpers ready.")

Inference helpers ready.


In [12]:
# ── Cell 5: Sliding Window Real-Time Inference ───────────────────────────────
#
# Logic:
#   - Maintain a deque of the last 16 preprocessed frame tensors
#   - Every new frame: preprocess → append to deque (auto-drops oldest)
#   - Once deque is full (16 frames): run model → display prediction
#   - The window slides by 1 frame every tick — dense predictions
#
# Press Q to quit.

frame_buffer = deque(maxlen=CFG["seq_len"])   # sliding window

cap = cv2.VideoCapture(CFG["cam_index"])
if not cap.isOpened():
    raise RuntimeError(
        f"Cannot open camera index {CFG['cam_index']}. "
        f"Try changing CFG['cam_index'] to 1 or 2."
    )

# Set resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CFG["display_w"])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CFG["display_h"])

print("Webcam opened. Starting inference loop...")
print("Press Q in the video window to quit.")

# State
current_label  = "..."
current_conf   = 0.0
current_probs  = None
top_preds      = [("...", 0.0)] * 3
fps            = 0.0
prev_time      = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame — camera disconnected?")
        break

    # ── FPS calculation ──
    now      = time.time()
    fps      = 1.0 / max(now - prev_time, 1e-6)
    prev_time = now

    # ── Preprocess and push to sliding window ──
    tensor = preprocess_frame(frame)   # (3, 128, 128)
    frame_buffer.append(tensor)        # deque auto-drops oldest when full

    # ── Predict once window is full ──
    if len(frame_buffer) == CFG["seq_len"]:
        current_label, current_conf, current_probs = predict_from_window(frame_buffer)

        # Build top-3 list
        sorted_indices = current_probs.argsort(descending=True)
        top_preds = [(idx2class[i.item()], current_probs[i].item())
                     for i in sorted_indices[:3]]

    # ── Draw overlay on frame ──
    frame = draw_overlay(
        frame,
        label       = current_label,
        confidence  = current_conf,
        buffer_size = len(frame_buffer),
        fps         = fps,
        top_preds   = top_preds,
        conf_thresh = CFG["conf_thresh"],
    )

    cv2.imshow("ASL Real-Time Inference  |  Press Q to quit", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Quitting...")
        break

cap.release()
cv2.destroyAllWindows()
print("Done.")

Webcam opened. Starting inference loop...
Press Q in the video window to quit.
Quitting...
Done.
